# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [145]:
# Your solution goes here
import requests
import pandas as pd
import time

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


In [146]:
#Connection
session = requests.Session()

retry_strategy = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504]
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("https://", adapter)
session.mount("http://", adapter)

BASE_URL = "https://books.toscrape.com/"

In [147]:
#Access the website
response = session.get(BASE_URL, timeout=20)

response.raise_for_status()
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

print("Status:", response.status_code)
print("Website:", soup.title.get_text(strip=True))

Status: 200
Website: All products | Books to Scrape - Sandbox


In [148]:
#Identify the books
books = soup.find_all(
    "article",
    class_="product_pod"
)

print("Books found:", len(books))

Books found: 20


In [149]:
#Extract title, price and rating
first_book = books[0]

# Title
title = first_book.find("h3").find("a")["title"]

# Price
price_text = first_book.find(
    "p",
    class_="price_color"
).get_text(strip=True)

price = float(price_text.replace("£", ""))

# Rating
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

rating_word = first_book.find(
    "p",
    class_="star-rating"
)["class"][1]

rating = rating_map[rating_word]

print("Title:", title)
print("Price:", price)
print("Rating:", rating)

Title: A Light in the Attic
Price: 51.77
Rating: 3


In [150]:
#Access the individual product page
book_link = first_book.find("h3").find("a")["href"]

book_url = urljoin(BASE_URL, book_link)

print("Book URL:", book_url)

book_response = session.get(book_url, timeout=20)
book_response.raise_for_status()
book_response.encoding = "utf-8"

book_soup = BeautifulSoup(
    book_response.text,
    "html.parser"
)

print("Product:", book_soup.h1.get_text(strip=True))

Book URL: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
Product: A Light in the Attic


In [151]:
#Extract UPC, genre, availability and description
# UPC and availability
table = book_soup.find("table")

product_info = {}

for row in table.find_all("tr"):
    key = row.find("th").get_text(strip=True)
    value = row.find("td").get_text(strip=True)

    product_info[key] = value

upc = product_info["UPC"]

availability = (
    "In stock" in product_info["Availability"]
)

# Genre
breadcrumb = book_soup.find(
    "ul",
    class_="breadcrumb"
)

items = breadcrumb.find_all("li")

genre = items[2].get_text(strip=True)

# Description
description_section = book_soup.find(
    "div",
    id="product_description"
)

description = "No description available"

if description_section:
    paragraph = description_section.find_next_sibling("p")

    if paragraph:
        description = paragraph.get_text(strip=True)

# Display results
print("UPC:", upc)
print("Genre:", genre)
print("Availability:", availability)
print("Description:", description)

UPC: a897fe39b1053632
Genre: Poetry
Availability: True
Description: It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,And your cradle, too?Baby, I

In [152]:
#Create a test DataFrame
book_data = {
    "UPC": upc,
    "Title": title,
    "Price (£)": price,
    "Rating": rating,
    "Genre": genre,
    "Availability": availability,
    "Description": description
}

df_test = pd.DataFrame([book_data])

display(df_test)

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,a897fe39b1053632,A Light in the Attic,51.77,3,Poetry,True,It's hard to imagine a world without A Light i...


In [153]:
#final function
def scrape_books(min_rating, max_price):

    # 1. Initialize variables
    filtered_books = []

    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    columns = [
        "UPC",
        "Title",
        "Price (£)",
        "Rating",
        "Genre",
        "Availability",
        "Description"
    ]

    current_url = BASE_URL
    page_count = 0

    # 2. Visit catalogue pages
    while current_url:

        page_count += 1
        print("Processing page:", page_count)

        time.sleep(2)

        response = session.get(
            current_url,
            timeout=20
        )

        response.raise_for_status()
        response.encoding = "utf-8"

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        books = soup.find_all(
            "article",
            class_="product_pod"
        )

        # 3. Process each book
        for book in books:

            # Extract title
            title = book.find("h3").find("a")["title"]

            # Extract price
            price_text = book.find(
                "p",
                class_="price_color"
            ).get_text(strip=True)

            price = float(
                price_text.replace("£", "")
            )

            # Extract rating
            rating_word = book.find(
                "p",
                class_="star-rating"
            )["class"][1]

            rating = rating_map[rating_word]

            # 4. Apply filters
            if rating >= min_rating and price <= max_price:

                # 5. Visit the individual product page
                book_link = book.find("h3").find("a")["href"]

                book_url = urljoin(
                    current_url,
                    book_link
                )

                time.sleep(2)

                book_response = session.get(
                    book_url,
                    timeout=20
                )

                book_response.raise_for_status()
                book_response.encoding = "utf-8"

                book_soup = BeautifulSoup(
                    book_response.text,
                    "html.parser"
                )

                # 6. Extract UPC and availability
                table = book_soup.find("table")

                product_info = {}

                for row in table.find_all("tr"):
                    key = row.find("th").get_text(strip=True)
                    value = row.find("td").get_text(strip=True)

                    product_info[key] = value

                upc = product_info["UPC"]

                availability = (
                    "In stock" in product_info["Availability"]
                )

                # 7. Extract genre
                breadcrumb = book_soup.find(
                    "ul",
                    class_="breadcrumb"
                )

                items = breadcrumb.find_all("li")

                genre = items[2].get_text(strip=True)

                # 8. Extract description
                description_section = book_soup.find(
                    "div",
                    id="product_description"
                )

                description = "No description available"

                if description_section:
                    paragraph = description_section.find_next_sibling("p")

                    if paragraph:
                        description = paragraph.get_text(strip=True)

                # 9. Store the qualifying book
                book_data = {
                    "UPC": upc,
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Genre": genre,
                    "Availability": availability,
                    "Description": description
                }

                filtered_books.append(book_data)

        # 10. Find the next catalogue page
        next_button = soup.select_one("li.next a")

        if next_button:
            current_url = urljoin(
                current_url,
                next_button["href"]
            )
        else:
            current_url = None

    # 11. Create and return the DataFrame
    df = pd.DataFrame(
        filtered_books,
        columns=columns
    )

    print("Pages processed:", page_count)
    print("Matching books:", len(df))

    return df

In [154]:
#Running the function
df = scrape_books(
    min_rating=4,
    max_price=20
)

display(df.head(10))

print("Total matching books:", len(df))
print("DataFrame shape:", df.shape)

Processing page: 1
Processing page: 2
Processing page: 3
Processing page: 4
Processing page: 5
Processing page: 6
Processing page: 7
Processing page: 8
Processing page: 9
Processing page: 10
Processing page: 11
Processing page: 12
Processing page: 13
Processing page: 14
Processing page: 15
Processing page: 16
Processing page: 17
Processing page: 18
Processing page: 19
Processing page: 20
Processing page: 21
Processing page: 22
Processing page: 23
Processing page: 24
Processing page: 25
Processing page: 26
Processing page: 27
Processing page: 28
Processing page: 29
Processing page: 30
Processing page: 31
Processing page: 32
Processing page: 33
Processing page: 34
Processing page: 35
Processing page: 36
Processing page: 37
Processing page: 38
Processing page: 39
Processing page: 40
Processing page: 41
Processing page: 42
Processing page: 43
Processing page: 44
Processing page: 45
Processing page: 46
Processing page: 47
Processing page: 48
Processing page: 49
Processing page: 50
Pages pro

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,True,Aaron Ledbetter’s future had been planned out ...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,True,"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,True,A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,True,"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,True,"Every summer, Rose goes with her mom and dad t..."
5,709822d0b5bcb7f4,Thirst,17.27,5,Fiction,True,"On a searing summer Friday, Eddie Chapman has ..."
6,0fa6dceead7ce47a,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5,Sequential Art,True,THE LONG-AWAITED STORY OF FANGIRLS TAKING ON T...
7,0e691eda369f4e09,Princess Between Worlds (Wide-Awake Princess #5),13.34,5,Fantasy,True,Just as Annie and Liam are busy making plans t...
8,889139b8e9c4cb36,"Outcast, Vol. 1: A Darkness Surrounds Him (Out...",15.44,4,Sequential Art,True,NEW HORROR SERIES FROM THE WALKING DEAD CREATO...
9,72f9d5be3472d34e,Mama Tried: Traditional Italian Cooking for th...,14.02,4,Food and Drink,True,Cecilia Granata grew up cooking with her famil...


Total matching books: 75
DataFrame shape: (75, 7)


In [155]:
#Check final data frame

# Check DataFrame structure
print("Shape:", df.shape)

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check duplicate books
print("\nDuplicate UPCs:")
print(df["UPC"].duplicated().sum())

# Validate filtering conditions
if not df.empty:

    print("\nMaximum price:")
    print(df["Price (£)"].max())

    print("\nMinimum rating:")
    print(df["Rating"].min())

    print("\nPrice filter:")
    print((df["Price (£)"] <= 20).all())

    print("\nRating filter:")
    print((df["Rating"] >= 4).all())

else:
    print("No matching books were collected.")

Shape: (75, 7)

Missing values:
UPC             0
Title           0
Price (£)       0
Rating          0
Genre           0
Availability    0
Description     0
dtype: int64

Duplicate UPCs:
0

Maximum price:
19.69

Minimum rating:
4

Price filter:
True

Rating filter:
True


In [156]:
#Validating result

# Check DataFrame structure
print("Shape:", df.shape)

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check duplicate books
print("\nDuplicate UPCs:")
print(df["UPC"].duplicated().sum())

# Validate filtering conditions
if not df.empty:

    print("\nMaximum price:")
    print(df["Price (£)"].max())

    print("\nMinimum rating:")
    print(df["Rating"].min())

    print("\nPrice filter:")
    print((df["Price (£)"] <= 20).all())

    print("\nRating filter:")
    print((df["Rating"] >= 4).all())

else:
    print("No matching books were collected.")

Shape: (75, 7)

Missing values:
UPC             0
Title           0
Price (£)       0
Rating          0
Genre           0
Availability    0
Description     0
dtype: int64

Duplicate UPCs:
0

Maximum price:
19.69

Minimum rating:
4

Price filter:
True

Rating filter:
True


In [157]:
#To analyze and export the results
genre_summary = df.groupby("Genre").agg(
    Number_of_Books=("UPC", "count"),
    Average_Price=("Price (£)", "mean"),
    Average_Rating=("Rating", "mean")
)

genre_summary = genre_summary.sort_values(
    by="Number_of_Books",
    ascending=False
)

display(genre_summary.round(2))

,Number_of_Books,Average_Price,Average_Rating
Genre,,,
Default,10,14.88,4.70
Sequential Art,9,15.24,4.56
Young Adult,7,13.74,4.71
Nonfiction,6,14.70,4.50
Fiction,5,13.16,4.60
Poetry,4,15.34,4.50
Food and Drink,4,12.20,4.50
Fantasy,3,14.24,4.67
Romance,3,15.65,4.33


In [158]:
df.to_csv(
    "books_scraping_results.csv",
    index=False,
    encoding="utf-8-sig"
)

genre_summary.to_csv(
    "books_genre_summary.csv",
    encoding="utf-8-sig"
)

print("CSV files created successfully!")

CSV files created successfully!
